# 02 — Homography: camera image → Room 204 floor map

Step A6. Saves each camera's first frame, holds the (empty) calibration points, computes
`H` with `cv2.findHomography` once four point pairs are filled in, and maps a track box's feet onto the map.

**To calibrate a camera:** open `data/calibration/<cam_id>_frame.png`, pick four points on the floor
(e.g. tile corners, door frame bottom), note their pixel `(x, y)`, find the same four points on the
Room 204 map, write both lists below in the same order and rerun the notebook.

In [1]:
import json, sys
from pathlib import Path

import cv2
import numpy as np

REPO = Path.cwd().resolve()
while not (REPO / "data" / "video_config.json").exists():
    REPO = REPO.parent
sys.path.insert(0, str(REPO / "ml" / "tracking"))
from paths import camera_ids, configured_take, video_path

TAKE = configured_take()
CAMS = camera_ids()
CALIB_DIR = REPO / "data" / "calibration"
CALIB_DIR.mkdir(parents=True, exist_ok=True)
print("take", TAKE, "| cameras", CAMS)

take 4 | cameras ['cam_1', 'cam_2', 'attendance_cam']


## 1. First frame of each camera

In [2]:
FRAME_SIZE = {}  # cam_id -> (width, height) in pixels
for cam in CAMS:
    src = video_path(cam, TAKE)
    cap = cv2.VideoCapture(str(src))
    ok, frame = cap.read()
    cap.release()
    assert ok, f"could not read the first frame of {src.name}"
    out = CALIB_DIR / f"{cam}_frame.png"
    cv2.imwrite(str(out), frame)
    h, w = frame.shape[:2]
    FRAME_SIZE[cam] = (w, h)
    check = cv2.imread(str(out))
    print(f"{cam}: {src.name} -> {out.relative_to(REPO)}  {w}x{h}  re-read ok={check is not None and check.shape == frame.shape}")

cam_1: cam_1_take4.mp4 -> data/calibration/cam_1_frame.png  1280x720  re-read ok=True


cam_2: cam_2_take4.mp4 -> data/calibration/cam_2_frame.png  1280x720  re-read ok=True
attendance_cam: attendance_cam_take4.mp4 -> data/calibration/attendance_cam_frame.png  1280x720  re-read ok=True


## 2. Calibration points (fill in by hand)

In [3]:
# image_points: pixel coordinates (x, y) of four floor points in the camera image
#               (data/calibration/<cam_id>_frame.png; x to the right, y down, origin top-left).
# map_points:   the same four points on the Room 204 map, normalised to [0, 1]
#               (x to the right, y down). The corridor region is x < 0.3.
# Same order in both lists. Four pairs give an exact fit; more pairs give a least-squares fit.
# Candidate floor landmarks found in the take-4 first frames (AI draft, pixel (x, y); verify on the PNG).
# They are NOT enough on their own: each camera shows only one or two wall-floor lines, so no four fixed
# floor points without three on one line exist. Add floor markers (e.g. 4 tape crosses at measured spots)
# visible to cam_1 and cam_2, then fill CALIB with those.
#   cam_1: classroom door, floor at left jamb (724, 581); floor at right jamb (868, 548)  - both on the corridor wall base
#   cam_2: doorway, floor at left jamb (248, 590); floor at right jamb (433, 590)         - both on the corridor wall base
#   attendance_cam: no floor visible (camera looks up at the door frame) -> no homography possible from this view
# The mock map in data/mock/session.json is a placeholder layout, not measured, so map_points are not guessed from it.
CALIB = {
    "cam_1":          {"image_points": [], "map_points": []},
    "cam_2":          {"image_points": [], "map_points": []},
    "attendance_cam": {"image_points": [], "map_points": []},
}
CORRIDOR_MAX_X = 0.3

## 3. Homography

In [4]:
def compute_homography(cam, points=CALIB):
    """H (3x3) from the calibration points of `cam`, saved to data/calibration/<cam>_homography.json.
    Returns None (and saves nothing) while fewer than four valid pairs are filled in."""
    img = np.asarray(points[cam]["image_points"], dtype=np.float64).reshape(-1, 2)
    mp = np.asarray(points[cam]["map_points"], dtype=np.float64).reshape(-1, 2)
    if len(img) < 4 or len(img) != len(mp):
        print(f"{cam}: skipped, need >= 4 pairs with matching counts (have {len(img)} image, {len(mp)} map)")
        return None
    if not ((mp >= 0) & (mp <= 1)).all():
        print(f"{cam}: skipped, map_points must be normalised to [0, 1]")
        return None
    H, _ = cv2.findHomography(img, mp, 0)  # 0 = plain least squares over all pairs
    if H is None:
        print(f"{cam}: skipped, points are degenerate (e.g. three on one line)")
        return None
    out = CALIB_DIR / f"{cam}_homography.json"
    out.write_text(json.dumps({"cam_id": cam, "take": TAKE, "frame_size": FRAME_SIZE.get(cam),
                               "image_points": img.tolist(), "map_points": mp.tolist(),
                               "H": H.tolist()}, indent=1), encoding="utf-8")
    print(f"{cam}: H saved to {out.relative_to(REPO)}")
    return H

H = {cam: compute_homography(cam) for cam in CAMS}

cam_1: skipped, need >= 4 pairs with matching counts (have 0 image, 0 map)
cam_2: skipped, need >= 4 pairs with matching counts (have 0 image, 0 map)
attendance_cam: skipped, need >= 4 pairs with matching counts (have 0 image, 0 map)


## 4. Box → map

Track boxes follow the data contract: `[x, y, w, h]` as fractions of the frame, `x, y` = **top-left**.
The feet are the bottom-centre: `((x + w/2) * width, (y + h) * height)` in pixels, the same space as `image_points`.

In [5]:
def foot_to_map(box, H, frame_size=(1280, 720)):
    """Map a normalised top-left [x, y, w, h] box to (map_x, map_y) on the Room 204 map.
    frame_size = (width, height) in pixels of the image the points were picked on (all take-4 cameras are 1280x720)."""
    if H is None:
        raise ValueError("no homography for this camera yet: fill in its calibration points")
    x, y, w, h = box
    fw, fh = frame_size
    foot = np.array([[[(x + w / 2) * fw, (y + h) * fh]]], dtype=np.float64)
    mx, my = cv2.perspectiveTransform(foot, np.asarray(H, dtype=np.float64))[0, 0]
    return float(mx), float(my)

def in_corridor(map_xy):
    return map_xy[0] < CORRIDOR_MAX_X

## 5. Checks (synthetic points only — not calibration data)

In [6]:
# Placeholders must still be empty and missing points must not crash anything.
assert all(not v["image_points"] and not v["map_points"] for v in CALIB.values()), "calibration lists are not empty"
assert all(h is None for h in H.values())
assert not any((CALIB_DIR / f"{c}_homography.json").exists() for c in CAMS)
try:
    foot_to_map([0.4, 0.5, 0.2, 0.3], H["cam_1"])
except ValueError as e:
    print("missing H handled:", e)
print("frames saved:", sorted(p.name for p in CALIB_DIR.glob("*_frame.png")))

# Synthetic 1: image corners -> unit square. The feet of [0.4, 0.5, 0.2, 0.3] are at (0.5, 0.8).
W, Hh = 1280, 720
syn_img = [[0, 0], [W, 0], [W, Hh], [0, Hh]]
syn_map = [[0, 0], [1, 0], [1, 1], [0, 1]]
Hs, _ = cv2.findHomography(np.float64(syn_img), np.float64(syn_map), 0)
p = foot_to_map([0.4, 0.5, 0.2, 0.3], Hs, (W, Hh))
assert np.allclose(p, (0.5, 0.8)), p
print("synthetic scale:", np.round(p, 6))

# Synthetic 2: a perspective trapezoid (far edge narrower) -> unit square; the 4 points map exactly,
# and a box whose feet sit on the bottom-left source corner lands at (0, 1).
trap = [[400, 300], [880, 300], [1280, 720], [0, 720]]
Ht, _ = cv2.findHomography(np.float64(trap), np.float64(syn_map), 0)
back = cv2.perspectiveTransform(np.float64(trap).reshape(-1, 1, 2), Ht).reshape(-1, 2)
assert np.allclose(back, syn_map, atol=1e-9), back
p2 = foot_to_map([-0.05, 0.6, 0.1, 0.4], Ht, (W, Hh))  # feet at pixel (0, 720)
assert np.allclose(p2, (0, 1), atol=1e-9), p2
print("synthetic perspective:", np.round(p2, 6), "| corridor?", in_corridor(p2))

# Synthetic 3: compute_homography() rejects bad input without crashing or writing files.
for bad in ({"image_points": syn_img[:3], "map_points": syn_map[:3]},
            {"image_points": syn_img, "map_points": [[0, 0], [2, 0], [2, 2], [0, 2]]}):
    assert compute_homography("cam_1", {"cam_1": bad}) is None
assert not (CALIB_DIR / "cam_1_homography.json").exists()
print("all checks passed")

missing H handled: no homography for this camera yet: fill in its calibration points
frames saved: ['attendance_cam_frame.png', 'cam_1_frame.png', 'cam_2_frame.png']
synthetic scale: [0.5 0.8]
synthetic perspective: [-0.  1.] | corridor? True
cam_1: skipped, need >= 4 pairs with matching counts (have 3 image, 3 map)
cam_1: skipped, map_points must be normalised to [0, 1]
all checks passed
